In [ ]:
import os
from pathlib import Path


# Change directory
# Modify this cell to insure that the output shows the correct path.
# Define all paths relative to the project root shown in the cell output
# project_root = "/Users/jackienguyen/Desktop/DLVR/freqtrade"
project_root = "/workspaces/freqtrade"

i = 0
try:
    os.chdir(project_root)
    if not Path("LICENSE").is_file():
        i = 0
        while i < 4 and (not Path("LICENSE").is_file()):
            os.chdir(Path(Path.cwd(), "../"))
            i += 1
        project_root = Path.cwd()
except FileNotFoundError:
    logging.info("Please define the project root relative to the current directory")
logging.info(Path.cwd())

/workspaces/freqtrade


# Scrape github repo relate to freqtrade
-  like explore the whole working related space

In [ ]:

# Description:
# This script scrapes GitHub repositories and extracts strategy-related Python files based on specific keywords.
# It uses the GitHub API to search for repositories and code, processes the repositories to identify relevant files,
# and saves the metadata. The script handles rate limits dynamically, excludes virtual environments, and cleans up
# temporary directories to ensure efficient and organized operation.

# !pip install gitpython
from datetime import datetime, timedelta, timezone
from github import Github
from pathlib import Path
import os
import json
import time
import shutil
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from git import Repo
import logging

# Initialize logger
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# Initialize GitHub Client
GITHUB_TOKEN = ""
g = Github(GITHUB_TOKEN)

# Directory to store results
BASE_DIR = Path(project_root)/"user_data/strategies/github_scraped_strategies"
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Metadata file to track processed repositories
METADATA_FILE = BASE_DIR / "metadata.json"
if not METADATA_FILE.exists():
    with METADATA_FILE.open("w") as f:
        # Prepopulate metadata with excluded repositories
        json.dump({
            "freqtrade/freqtrade": {
                "last_pushed": "9999-12-31T23:59:59",
                "url": "https://github.com/freqtrade/freqtrade",
                "strategy_files": []
            },
            "Fourseascocreation/ccxt": {
                "last_pushed": "9999-12-31T23:59:59",
                "url": "https://github.com/Fourseascocreation/ccxt",
                "strategy_files": []
            },
        }, f, indent=2)

# Load existing metadata
with METADATA_FILE.open("r") as f:
    processed_repos = json.load(f)

# def save_metadata():
#     """Save the processed_repos metadata to the file."""
#     with METADATA_FILE.open("w") as f:
#         json.dump(processed_repos, f, indent=2)


def save_metadata():
    """Save the processed_repos metadata to the file, ensuring specific timestamps are at the bottom."""
    sorted_repos = dict(sorted(
        processed_repos.items(),
        key=lambda item: (item[1]["last_pushed"] == "9999-12-31T23:59:59", item[1]["last_pushed"])
    ))

    with METADATA_FILE.open("w") as f:
        json.dump(sorted_repos, f, indent=2)


def handle_rate_limit():
    """Check and handle GitHub API rate limits dynamically."""
    remaining = int(g.get_rate_limit().core.remaining)
    if remaining == 0:
        reset_time = g.get_rate_limit().core.reset.timestamp()
        sleep_time = max(0, reset_time - time.time())
        logger.info(f"Rate limit reached. Sleeping for {sleep_time} seconds.")
        time.sleep(sleep_time)

# def cleanup_directory(path):
#     """Remove empty directories in the given path."""
#     for root, dirs, files in os.walk(path, topdown=False):
#         for dir_name in dirs:
#             dir_path = Path(root) / dir_name
#             if not any(dir_path.iterdir()):
#                 dir_path.rmdir()

def search_candidates(query, search_type="repositories"):
    """Search GitHub for repositories or code using a query."""
    if search_type == "repositories":
        return g.search_repositories(query)
    elif search_type == "code":
        return g.search_code(query)

# def process_file(file_path, strategy_files):
#     """Process a single file to determine if it is a strategy file."""
#     try:
#         with file_path.open("r", encoding="utf-8") as f:
#             content = f.read()
#             if "(IStrategy)" in content and "freqtrade" in content:
#                 strategy_files.append(str(file_path))
#             else:
#                 file_path.unlink()  # Remove non-strategy Python files
#     except Exception as e:
#         logger.error(f"Failed to process file {file_path}: {e}")
#         if file_path.exists():
#             file_path.unlink()  # Remove problematic files

# def walk_directory(directory, strategy_files):
#     """Walk through the repository directory and process files."""
#     directories_to_prioritize = ["user_data", "strategy", "strategies"]

#     with ThreadPoolExecutor(max_workers=os.cpu_count() * 2) as executor:
#         futures = []
#         for root, dirs, files in os.walk(directory):
#             # Ignore venv folders
#             dirs[:] = [d for d in dirs if d != "venv"]

#             # Prioritize specific directories
#             dirs.sort(key=lambda d: 0 if d in directories_to_prioritize else 1)

#             for file in files:
#                 file_path = Path(root) / file
#                 if file.endswith(".py"):
#                     futures.append(executor.submit(process_file, file_path, strategy_files))
#                 else:
#                     file_path.unlink()  # Remove non-Python files

#         # Wait for all futures to complete
#         for future in futures:
#             future.result()

def process_repository(repo, repo_name):
    """Clone the repository, identify strategy modules, and clean up unnecessary files."""
    logger.info(f"Processing repository: {repo_name}")

    # Skip if already processed and no updates
    if repo.stargazers_count < 2:
        logger.info(f"Skipping {repo_name}, repository has no stars.")
        return

    if repo_name in processed_repos and repo.pushed_at <= datetime.fromisoformat(processed_repos[repo_name]["last_pushed"]):
        logger.info(f"Skipping {repo_name}, no updates since last processed.")
        return

    # Skip if last commit is older than two years
    one_years_ago = datetime.now(timezone.utc) - timedelta(days=1 * 365)  # Ensure timezone-aware datetime
    if repo.pushed_at < one_years_ago:
        logger.info(f"Skipping {repo_name}, last commit is older than two years.")
        return

    # Skip repositories with less than 2 commits
    try:
        commit_count = repo.get_commits().totalCount
        if commit_count < 2:
            logger.info(f"Skipping {repo_name}, repository has less than 2 commits.")
            return
    except Exception as e:
        logger.error(f"Failed to retrieve commit count for {repo_name}: {e}")
        return

    # Clone the repository
    repo_dir = BASE_DIR / repo_name.replace("/", "_")
    try:
        logger.info(f"Cloning repository: {repo.clone_url} into {repo_dir}")
        Repo.clone_from(repo.clone_url, str(repo_dir), depth=1)
    except Exception as e:
        if "already exists and is not an empty directory" in str(e):
            logger.warning(f"Target directory {repo_dir} is not empty. Removing and retrying.")
            shutil.rmtree(repo_dir)
            try:
                Repo.clone_from(repo.clone_url, str(repo_dir), depth=1)
            except Exception as retry_e:
                logger.error(f"Retry failed to clone {repo_name}: {retry_e}")
                return
        else:
            logger.error(f"Failed to clone {repo_name}: {e}")
            return

    # strategy_files = []
    # walk_directory(repo_dir, strategy_files)

    # Save metadata for the processed repository
    processed_repos[repo_name] = {
        "last_pushed": repo.pushed_at.isoformat(),
        "url": repo.html_url,
        # "strategy_files": strategy_files,
    }
    # logger.info(f"Processed repository {repo_name} with {len(strategy_files)} strategy files.\n")

    # Cleanup: Remove empty folders
    # cleanup_directory(repo_dir)

def batch_save_metadata(counter, batch_size=10):
    """Save metadata after processing a batch of repositories."""
    if counter % batch_size == 0:
        save_metadata()

def main():
    # Search queries
    repo_query = "freqtrade in:name,description,readme language:python"
    # code_query = '"freqtrade" "(IStrategy)" in:file extension:py size:>1500'

    logger.info(f"Searching repositories for query: {repo_query}")
    repos = search_candidates(repo_query, "repositories")
    repo_count = 0
    batch_size = 10

    for i, repo in enumerate(repos):
        repo_name = repo.full_name
        logging.info(f"Repo {i+1}: {repo_name}")
        if repo_name not in processed_repos:
            handle_rate_limit()
            repo_count += 1
            process_repository(repo, repo_name)
            batch_save_metadata(i + 1, batch_size)
        else:
            logger.info(f"Skipping already processed repository: {repo_name}")

    logger.info(f"Total repositories processed: {repo_count}\n\n\n")

    # logger.info(f"Searching code for query: {code_query}")
    # codes = search_candidates(code_query, "code")
    # code_repo_count = 0

    # for i, code in enumerate(codes):
    #     repo = code.repository
    #     repo_name = repo.full_name
    #     logger.info(f"Processing code file {i + 1}: {repo_name}")
    #     if repo_name not in processed_repos:
    #         handle_rate_limit()
    #         code_repo_count += 1
    #         process_repository(repo, repo_name)
    #         batch_save_metadata(i + 1, batch_size)
    #     else:
    #         logger.info(f"Skipping already processed repository: {repo_name}")

    # logger.info(f"Total code-related repositories processed: {code_repo_count}")

    # Final save after all processing
    save_metadata()
    logger.info("Processing completed.")

if __name__ == "__main__":
    main()


In [ ]:
from datetime import datetime, timedelta, timezone
import logging
from github import Github
import shutil
from pathlib import Path
import os
import shutil
from pathlib import Path
import re

# Initialize logger
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# Initialize GitHub Client
BASE_DIR = Path(project_root)/"user_data/strategies/github_scraped_strategies"
GITHUB_TOKEN = ""
g = Github(GITHUB_TOKEN)

def process_file(file_path):
    """Process a single file to modify imports and fix invalid escape sequences."""
    try:
        with file_path.open("r", encoding="utf-8") as f:
            content = f.readlines()

        modified_content = []

        # Improved regex to detect 'from keras.something import ...' and 'import keras.something'
        keras_import_pattern = re.compile(r'^\s*(from\s+keras(?:\.\w+)+\s+import\s+\w+|import\s+keras(?:\.\w+)+)')

        for line in content:
            if keras_import_pattern.match(line) and not line.startswith(" "):
                # Wrap the keras import with try-except
                modified_content.append("try:\n")
                modified_content.append(f"    {line}")
                modified_content.append("except ImportError:\n")
                modified_content.append("    pass  # Keras module is not installed\n\n")
            else:
                modified_content.append(line)

        modified_content = "".join(modified_content)

        # Modify specific import lines
        replacements = {
            "from freqtrade.strategy.hyper import IntParameter": "from freqtrade.strategy import IntParameter",
            "from freqtrade.strategy.hyper import CategoricalParameter": "from freqtrade.strategy import CategoricalParameter",
            "from freqtrade.strategy.hyper import DecimalParameter" : "from freqtrade.strategy import DecimalParameter",
            "from user_data." : """import sys, os\nfile_path = os.path.abspath(__file__)\nparts = file_path.split(os.sep)\nindex = parts.index("github_scraped_strategies")\ntarget_path = os.path.join(*parts[:index+2])\nsys.path.insert(0, '/' + target_path)\nfrom user_data.""",
        }
        for old, new in replacements.items():
            if (old == "from user_data.") and ("sys.path.insert(0, target_path)\nfrom user_data." in modified_content):
                pass
            else:
                modified_content = modified_content.replace(old, new)

        # Fix invalid escape sequences inside triple-quoted strings
        def fix_invalid_escapes(match):
            return match.group().replace(r"\.", ".")  # Replace invalid \. with .

        modified_content = re.sub(r'""".*?"""', fix_invalid_escapes, modified_content, flags=re.DOTALL)

        # Only write back if modifications were made
        if modified_content != content:
            with file_path.open("w", encoding="utf-8") as f:
                f.write(modified_content)
            logging.info(f"Fixed escape sequences and imports in {file_path}")

    except Exception as e:
        logger.error(f"Failed to process file {file_path}: {e}")


def walk_directory(directory):
    """Walk through the repository directory and process files, deleting .git folders."""
    directories_to_prioritize = ["user_data", "strategy", "strategies"]

    for root, dirs, files in os.walk(directory):
        # Ignore venv folders
        dirs[:] = [d for d in dirs if d != "venv"]

        # Prioritize specific directories
        dirs.sort(key=lambda d: 0 if d in directories_to_prioritize else 1)

        # Delete .git folders if found
        if ".git" in dirs:
            git_path = Path(root) / ".git"
            try:
                shutil.rmtree(git_path)
                logging.info(f"Deleted: {git_path}")
            except Exception as e:
                logging.info(f"Failed to delete {git_path}: {e}")
            dirs.remove(".git")  # Ensure it is not traversed further

        for file in files:
            file_path = Path(root) / file
            if file.endswith(".py"):
                process_file(file_path)

# Define the directory to process
directory = Path(project_root) / "user_data/strategies/github_scraped_strategies/"
walk_directory(directory)


In [1]:
# import shutil
# from pathlib import Path

# # List of folders to move
# to_move_folder = [
# # 'hamidreza07_freqai-strategy',
# # 'djienne_YOUTUBE_STRATEGIES_FREQTRADE'
# # 'Branly76_DecoBbDca'
# # 'frostaura_fa.services.plutus'
# # 'TheoBrigitte_freqtrade'
# # 'AlexCryptoKing_freqailstm'
# # 'AlexCryptoKing_freqtrade'
# # 'wtriantis_freqtrade_user_data'
# # 'fortesenselabs_trade_flow'
# # 'mrzdev_refreshpairlist'
# # 'imsatoshi_GeneTrader'
# 'Netanelshoshan_freqAI-LSTM'
# ]

# # Source and destination directories
# source_dir = Path(project_root) / "user_data/strategies/github_scraped_strategies"
# dest_dir = Path(project_root) / "user_data/strategies_backup"

# # Ensure the destination directory exists
# # dest_dir.mkdir(parents=True, exist_ok=True)

# # Move each folder
# for folder in to_move_folder:
#     src_path = source_dir / folder
#     dest_path = dest_dir / folder
#     if src_path.exists() and src_path.is_dir():
#         shutil.move(str(src_path), str(dest_path))
#         logging.info(f"Moved {src_path} to {dest_path}")
#     else:
#         logging.info(f"Folder {src_path} does not exist or is not a directory")

In [ ]:
# from pathlib import Path

# # Define the source directory
# source_dir = Path(project_root)/"user_data/strategies/github_scraped_strategies"

# # Count the number of directories
# remaining_folders = [f for f in source_dir.iterdir() if f.is_dir()]

# # logging.info the count of remaining folders
# logging.info(f"Number of remaining folders: {len(remaining_folders)}")

Number of remaining folders: 110


# Scrape strategy from https://strat.ninja/

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.edge.service import Service
import os
import requests
import shutil
import logging

# Setup the logging mechanism
logging.basicConfig(filename='strategy_scraping.log', level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

# Setup the WebDriver (Microsoft Edge in this case)
edge_driver_path = '/Users/jackienguyen/Desktop/msedgedriver'  # Update with the correct path
service = Service(edge_driver_path)
driver = webdriver.Edge(service=service)

# Define the URL and create a new list for strategy links
url = "https://strat.ninja/ranking.php"
strategy_links = []
new_links = []

# Create the directory for saving files
save_directory = '/Users/jackienguyen/Desktop/DLVR/freqtrade/user_data/strategy/strat_ninja_scraped_strategies'
if os.path.exists(save_directory):
    shutil.rmtree(save_directory)
os.makedirs(save_directory, exist_ok=True)

# Function to gather strategy links from the table
def gather_strategy_links():
    driver.get(url)
    wait = WebDriverWait(driver, 10)

    # Ensure "Hide Private" is checked and "Show all" is selected
    wait.until(EC.element_to_be_clickable((By.ID, "c7"))).click()
    rows_dropdown = wait.until(EC.element_to_be_clickable((By.NAME, "example_length")))
    rows_dropdown.click()
    wait.until(EC.element_to_be_clickable((By.XPATH, "//option[@value='-1']"))).click()

    # Wait for the page to update
    wait.until(EC.presence_of_element_located((By.XPATH, "//table[@id='example']//tr")))

    # Collect strategy links
    rows = driver.find_elements(By.XPATH, "//table[@id='example']//tr")
    for row in rows:
        try:
            strategy_link = row.find_element(By.XPATH, ".//th[2]/a").get_attribute("href")
            if strategy_link:
                strategy_links.append(strategy_link)
        except Exception as e:
            logging.info(f"Error extracting strategy link: {e}")
            continue  # Ignore rows where the link is not found

    logging.info(f"Collected {len(strategy_links)} strategy links.")
    return strategy_links

# Function to check if the content is HTML
def is_html(content, headers):
    # Check if the content type is HTML (either from headers or content itself)
    if 'text/html' in headers.get('Content-Type', ''):
        return True
    return "<html" in content.lower() or "<!DOCTYPE html>" in content.lower()

# Function to download and save the file (after checks)
def download_and_save_file(response, strategy_name, local_path):
    try:
        # response = requests.get(new_link)
        with open(local_path, 'wb') as f:
            f.write(response.content)
        logging.info(f"Saved file: {strategy_name}")
    except Exception as e:
        logging.error(f"Error saving {strategy_name}: {e}")

# Function to process each strategy link and gather download links
def process_strategy_links(strategy_links):
    for link in strategy_links:
        try:
            logging.info(f"Processing link: {link}")
            strategy_name = f"{link.split('=')[-1]}.py"
            local_path = os.path.join(save_directory, strategy_name)
            new_link = f"https://strat.ninja/mirror/{strategy_name}"

            # Check the status code and content type before calling the save function
            response = requests.get(new_link)
            if (response.status_code == 200) and not is_html(response.text, response.headers):
                    download_and_save_file(response, strategy_name, local_path)
            else:
                logging.error(f"Failed to fetch file: {new_link}")
                # If the direct link failed, try Selenium method
                logging.info(f"Attempting to fetch file using Selenium for {link}")
                driver.execute_script(f"window.open('{link}', '_blank');")
                driver.switch_to.window(driver.window_handles[-1])
                wait = WebDriverWait(driver, 10)

                target_element = wait.until(EC.element_to_be_clickable(
                    (By.XPATH, "/html/body/div[1]/div[2]/div[2]//a[.//img[@src='/images/ext_link.png']]")))
                new_link = target_element.get_attribute('href')
                driver.close()
                driver.switch_to.window(driver.window_handles[0])

                # Try downloading the file after getting the link via Selenium
                response = requests.get(new_link)
                if response.status_code == 200:
                    if is_html(response.text, response.headers):  # Check if the content is HTML

                        # Open new tab to access the code
                        driver.execute_script(f"window.open('{new_link}', '_blank');")
                        driver.switch_to.window(driver.window_handles[-1])

                        # Wait and locate the textarea containing the code
                        wait = WebDriverWait(driver, 10)
                        try:
                            code_element = wait.until(EC.presence_of_element_located((By.ID, "read-only-cursor-text-area")))
                            code_content = code_element.get_attribute("value")

                            # Save the extracted code to the local file
                            with open(local_path, 'w', encoding='utf-8') as f:
                                f.write(code_content)
                            logging.info(f"Saved file: {strategy_name} from textarea content")

                        except Exception as e:
                            logging.error(f"Failed to extract code from {new_link}: {e}")

                        # Close the tab and return to the main window
                        driver.close()
                        driver.switch_to.window(driver.window_handles[0])
                    else:
                        download_and_save_file(response, strategy_name, local_path)
                else:
                    logging.error(f"Failed to fetch file: {new_link}")
        except Exception as e:
            logging.info(f"Error processing {link}: {e}")

# Main execution flow
strategy_links = gather_strategy_links()
process_strategy_links(strategy_links)

# Close the browser
driver.quit()

In [ ]:
import os

def is_html_file(content):
    # Check for common HTML tags that indicate the file is an HTML file
    return "<html" in content.lower() and "<head>" in content.lower()

def get_html_files_list(directory):
    html_files = []

    # Loop through all files in the specified directory
    for file_name in os.listdir(directory):
        # Full file path
        file_path = os.path.join(directory, file_name)

        # Check if it's a file (not a directory)
        if os.path.isfile(file_path):
            try:
                # Read the contents of the file
                with open(file_path, 'r', encoding='utf-8') as file:
                    content = file.read()

                # Check if the file contains HTML content
                if is_html_file(content):
                    html_files.append(file_name)  # Append the filename to the list
            except Exception as e:
                print(f"Error reading {file_name}: {e}")

    return html_files

# Usage example:
directory = '/Users/jackienguyen/Desktop/DLVR/freqtrade/user_data/strategy/strat_ninja_scraped_strategies'
# directory = 'path_to_your_folder'  # Replace with your folder path
html_files = get_html_files_list(directory)

# Print the list of HTML file names
print(html_files)

[]
